# 01 -- Getting Started with Quantum Nexus

Sanity check notebook. Goals:

1. Verify Qiskit and all dependencies are installed
2. Connect to IBM Quantum Platform
3. Build and run a Bell state on the **Aer simulator**
4. Build and run a Bell state on **real IBM hardware**
5. Compare the results

If everything here works, the pipeline is solid and we can build on it.

## 1. Environment Check

In [ ]:
import qiskit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

print(f"Qiskit version:  {qiskit.__version__}")
print(f"Aer simulator:   available")
print(f"IBM Runtime:     available")
print(f"\nAll good. Let's build some circuits.")

## 2. Build a Bell State Circuit

The simplest entanglement demo: apply a Hadamard gate to qubit 0, then a CNOT
from qubit 0 to qubit 1. The result is the Bell state:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

When measured, you should see roughly 50/50 split between `00` and `11`, with
nothing else. If you see `01` or `10`, something is wrong (or your hardware
is noisy -- which is the whole reason NISQ research exists).

In [ ]:
from qiskit import QuantumCircuit

# Build the Bell state circuit
bell = QuantumCircuit(2, 2)
bell.h(0)       # Hadamard on qubit 0 -- creates superposition
bell.cx(0, 1)   # CNOT -- entangles qubit 1 with qubit 0
bell.measure([0, 1], [0, 1])

# Draw it
bell.draw("mpl")

## 3. Run on the Aer Simulator

Aer is a local classical simulator. No queue, no noise, instant results.
This is where we'll do most of our development work.

In [ ]:
from qiskit import transpile
from qiskit.visualization import plot_histogram

# Run on the local simulator
sim = AerSimulator()
transpiled = transpile(bell, sim)
sim_result = sim.run(transpiled, shots=1024).result()
sim_counts = sim_result.get_counts()

print(f"Simulator counts: {sim_counts}")
plot_histogram(sim_counts, title="Bell State -- Aer Simulator (1024 shots)")

## 4. Connect to IBM Quantum Platform

This uses the credentials saved by `scripts/setup_ibm_account.py`.
If this cell errors, re-run that script first.

In [ ]:
# Connect using saved credentials
service = QiskitRuntimeService()

# List available backends
backends = service.backends()
print("Available backends:")
for b in backends:
    status = b.status()
    print(f"  {b.name:30s}  qubits={b.num_qubits:>4d}  status={status.status_msg}")

## 5. Run on Real Quantum Hardware

Pick the least-busy backend and send the Bell circuit to actual qubits.
This will sit in a queue -- IBM's free tier shares hardware with everyone.
Results will have noise, which is the whole point of seeing both side by side.

In [ ]:
# Pick the least-busy backend with enough qubits
backend = service.least_busy(min_num_qubits=2, operational=True)
print(f"Selected backend: {backend.name} ({backend.num_qubits} qubits)")

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

# Transpile for the real backend's topology and gate set
transpiled_hw = transpile(bell, backend, optimization_level=1)

# Submit the job via the Sampler primitive
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_hw], shots=1024)

print(f"Job submitted: {job.job_id()}")
print(f"Job status:    {job.status()}")
print(f"\nThis may take a few minutes in the queue. Run the next cell to get results.")

In [ ]:
# Wait for the result and extract counts
hw_result = job.result()

# SamplerV2 returns a PubResult -- extract the bitstring counts
hw_counts = hw_result[0].data.meas.get_counts()
print(f"Hardware counts: {hw_counts}")

## 6. Compare: Simulator vs. Hardware

Side-by-side comparison. The simulator should show a perfect 50/50 split.
The hardware will show noise -- some `01` and `10` counts leaking in.
The gap between these two histograms is basically the whole NISQ problem.

In [ ]:
plot_histogram(
    [sim_counts, hw_counts],
    legend=["Simulator", f"Hardware ({backend.name})"],
    title="Bell State -- Simulator vs. Real Hardware (1024 shots)",
    figsize=(10, 6),
)

In [ ]:
# Quick fidelity check -- how close is hardware to the ideal result?
ideal = {"00": 512, "11": 512}
total_shots = sum(hw_counts.values())

# Fraction of counts in the "correct" outcomes (00 and 11)
correct = hw_counts.get("00", 0) + hw_counts.get("11", 0)
fidelity = correct / total_shots

print(f"Hardware fidelity: {fidelity:.3f}")
print(f"  |00>: {hw_counts.get('00', 0):>4d}  ({hw_counts.get('00', 0)/total_shots:.1%})")
print(f"  |11>: {hw_counts.get('11', 0):>4d}  ({hw_counts.get('11', 0)/total_shots:.1%})")
print(f"  |01>: {hw_counts.get('01', 0):>4d}  ({hw_counts.get('01', 0)/total_shots:.1%})  <- noise")
print(f"  |10>: {hw_counts.get('10', 0):>4d}  ({hw_counts.get('10', 0)/total_shots:.1%})  <- noise")
print(f"\nFidelity > 0.95 = good hardware day. < 0.90 = noisy.")

## 7. GHZ State -- Scaling Up

Let's try a 3-qubit GHZ state to see how noise scales with circuit size.
GHZ is the multi-qubit generalization of Bell:

$$|GHZ\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$$

More qubits = more CNOT gates = more noise accumulation. This is a quick
way to feel out the hardware's error characteristics.

In [ ]:
# Build a 3-qubit GHZ circuit
ghz = QuantumCircuit(3, 3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)
ghz.measure([0, 1, 2], [0, 1, 2])

ghz.draw("mpl")

In [ ]:
# Simulator run
ghz_transpiled_sim = transpile(ghz, sim)
ghz_sim_counts = sim.run(ghz_transpiled_sim, shots=1024).result().get_counts()

# Hardware run
ghz_transpiled_hw = transpile(ghz, backend, optimization_level=1)
ghz_job = sampler.run([ghz_transpiled_hw], shots=1024)
print(f"GHZ job submitted: {ghz_job.job_id()}")
print(f"Status: {ghz_job.status()}")

In [ ]:
# Get GHZ hardware results
ghz_hw_result = ghz_job.result()
ghz_hw_counts = ghz_hw_result[0].data.meas.get_counts()

plot_histogram(
    [ghz_sim_counts, ghz_hw_counts],
    legend=["Simulator", f"Hardware ({backend.name})"],
    title="GHZ State (3 qubits) -- Simulator vs. Hardware",
    figsize=(12, 6),
)

In [ ]:
# Compare fidelity: Bell (2 qubits) vs GHZ (3 qubits)
ghz_total = sum(ghz_hw_counts.values())
ghz_correct = ghz_hw_counts.get("000", 0) + ghz_hw_counts.get("111", 0)
ghz_fidelity = ghz_correct / ghz_total

print("Fidelity comparison:")
print(f"  Bell state (2 qubits, 1 CNOT):  {fidelity:.3f}")
print(f"  GHZ state  (3 qubits, 2 CNOTs): {ghz_fidelity:.3f}")
print(f"  Fidelity drop:                   {fidelity - ghz_fidelity:.3f}")
print(f"\nThis drop gives you a rough sense of per-gate error accumulation.")

## What's Next

Pipeline works. We can build circuits, run them locally, and send them to real hardware.

Next notebooks:
- **02**: Power system state estimation -- build the VQE circuits for real
- **03**: Anomaly detection -- quantum kernel methods on simulated sensor data

The fidelity numbers from this notebook are a useful baseline. When we start
building more complex circuits for actual problems, we'll know how much room
we have before noise eats the signal.